AVIRAL SAXENA | Roll No: 2025AG05387 | DM4ML Assignment 2

# Phase 6: Model Evaluation

This notebook evaluates and compares the popularity-based and personalized SVD recommendation models.

Evaluation metrics:
- Precision@10
- Recall@10
- NDCG@10
- Catalog Coverage
- Diversity
- RMSE
- MAE

In [ ]:
%pip install -q pandas numpy scikit-surprise mlflow

from google.colab import drive
drive.mount('/content/drive')

import os
import pickle
import numpy as np
import pandas as pd
import mlflow

PROJECT_DIR = "/content/drive/MyDrive/recommendation-pipeline"

PROCESSED_DATA_DIR = os.path.join(
    PROJECT_DIR, "data", "processed"
)

FEATURES_DIR = os.path.join(
    PROJECT_DIR, "data", "features"
)

MODEL_DIR = os.path.join(
    PROJECT_DIR, "models"
)

REPORT_DIR = os.path.join(
    PROJECT_DIR, "reports"
)

os.makedirs(REPORT_DIR, exist_ok=True)

train_df = pd.read_parquet(
    os.path.join(PROCESSED_DATA_DIR, "train_data.parquet")
)

test_df = pd.read_parquet(
    os.path.join(PROCESSED_DATA_DIR, "test_data.parquet")
)

movies_df = pd.read_parquet(
    os.path.join(PROCESSED_DATA_DIR, "movies.parquet")
)

with open(os.path.join(MODEL_DIR, "popularity_model.pkl"), "rb") as f:
    popularity_df = pickle.load(f)

with open(os.path.join(MODEL_DIR, "svd_model.pkl"), "rb") as f:
    svd_model = pickle.load(f)

print("Evaluation datasets and models loaded successfully!")
print("Training:", train_df.shape)
print("Testing:", test_df.shape)
print("Popularity model:", popularity_df.shape)

Mounted at /content/drive
Evaluation datasets and models loaded successfully!
Training: (80668, 13)
Testing: (20168, 13)
Popularity model: (3123, 3)


In [ ]:
from surprise import accuracy

testset = list(
    test_df[["userId", "movieId", "rating"]]
    .itertuples(index=False, name=None)
)

svd_predictions = svd_model.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

print("SVD RMSE:", round(svd_rmse, 4))
print("SVD MAE:", round(svd_mae, 4))

SVD RMSE: 1.0145
SVD MAE: 0.7827


In [ ]:
# evaluation settings

K = 10
RELEVANCE_THRESHOLD = 4.0

# Common eligible movie catalogue
eligible_movie_ids = set(
    popularity_df["movieId"].unique()
)

# Movies already rated by each user in training
user_seen_movies = (
    train_df.groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

# Relevant movies in the test set:
# test ratings of 4.0 or higher
relevant_movies = (
    test_df[test_df["rating"] >= RELEVANCE_THRESHOLD]
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

# Keep only eligible, previously unseen test movies
relevant_movies = {
    user: items & eligible_movie_ids - user_seen_movies.get(user, set())
    for user, items in relevant_movies.items()
}

# Evaluate users who have at least one relevant test movie
evaluation_users = sorted(
    user for user, items in relevant_movies.items()
    if len(items) > 0
)

print("Eligible movies:", len(eligible_movie_ids))
print("Users eligible for ranking evaluation:", len(evaluation_users))
print("Relevance threshold:", RELEVANCE_THRESHOLD)
print("K:", K)

Eligible movies: 3123
Users eligible for ranking evaluation: 114
Relevance threshold: 4.0
K: 10


## Evaluation Metric Functions

In [ ]:
def precision_at_k(recommended, relevant, k=10):
    hits = len(set(recommended[:k]) & relevant)
    return hits / k


def recall_at_k(recommended, relevant, k=10):
    if not relevant:
        return 0.0

    hits = len(set(recommended[:k]) & relevant)
    return hits / len(relevant)


def ndcg_at_k(recommended, relevant, k=10):
    recommended = recommended[:k]

    dcg = sum(
        1 / np.log2(rank + 1)
        for rank, movie_id in enumerate(recommended, start=1)
        if movie_id in relevant
    )

    ideal_hits = min(len(relevant), k)

    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    return dcg / idcg if idcg > 0 else 0.0


def catalog_coverage(all_recommendations, catalog):
    recommended_items = set(
        movie
        for recommendations in all_recommendations.values()
        for movie in recommendations[:K]
    )

    return len(recommended_items) / len(catalog)


print("Evaluation metric functions created successfully!")

Evaluation metric functions created successfully!


In [ ]:
# recommendations using popularity
popularity_recommendations = {}

ranked_popularity_movies = (
    popularity_df.sort_values(
        ["average_rating", "rating_count"],
        ascending=[False, False]
    )["movieId"].tolist()
)

for user_id in evaluation_users:

    seen = user_seen_movies.get(user_id, set())

    recommendations = [
        movie_id
        for movie_id in ranked_popularity_movies
        if movie_id not in seen
    ][:K]

    popularity_recommendations[user_id] = recommendations

print("Popularity recommendations generated!")
print("Example recommendations:")

first_user = evaluation_users[0]

print("User:", first_user)
print("Movie IDs:", popularity_recommendations[first_user])

Popularity recommendations generated!
Example recommendations:
User: 15
Movie IDs: [6460, 2239, 4334, 1041, 3068, 3201, 1104, 3451, 80906, 3152]


In [ ]:
# recommendations using SVD
svd_recommendations = {}

for user_id in evaluation_users:

    seen = user_seen_movies.get(user_id, set())

    candidate_movies = eligible_movie_ids - seen

    predictions = [
        (
            movie_id,
            svd_model.predict(user_id, movie_id).est
        )
        for movie_id in candidate_movies
    ]

    predictions.sort(
        key=lambda x: x[1],
        reverse=True
    )

    svd_recommendations[user_id] = [
        movie_id
        for movie_id, predicted_rating in predictions[:K]
    ]

print("SVD recommendations generated!")
print("Example recommendations:")

print("User:", first_user)
print("Movie IDs:", svd_recommendations[first_user])

SVD recommendations generated!
Example recommendations:
User: 15
Movie IDs: [np.int64(750), np.int64(318), np.int64(68157), np.int64(1104), np.int64(1945), np.int64(1223), np.int64(34405), np.int64(2329), np.int64(1208), np.int64(2858)]


In [ ]:
# calculating and comparing all ranking metrics
def evaluate_recommender(recommendations, relevant_movies, catalog):

    precision_scores = []
    recall_scores = []
    ndcg_scores = []

    for user_id in evaluation_users:

        recommended = recommendations.get(user_id, [])
        relevant = relevant_movies[user_id]

        precision_scores.append(
            precision_at_k(recommended, relevant, K)
        )

        recall_scores.append(
            recall_at_k(recommended, relevant, K)
        )

        ndcg_scores.append(
            ndcg_at_k(recommended, relevant, K)
        )

    return {
        "Precision@10": np.mean(precision_scores),
        "Recall@10": np.mean(recall_scores),
        "NDCG@10": np.mean(ndcg_scores),
        "Coverage": catalog_coverage(recommendations, catalog)
    }


popularity_metrics = evaluate_recommender(
    popularity_recommendations,
    relevant_movies,
    eligible_movie_ids
)

svd_metrics = evaluate_recommender(
    svd_recommendations,
    relevant_movies,
    eligible_movie_ids
)

comparison_df = pd.DataFrame([
    {
        "Model": "Popularity",
        **popularity_metrics,
        "RMSE": np.nan,
        "MAE": np.nan
    },
    {
        "Model": "SVD",
        **svd_metrics,
        "RMSE": svd_rmse,
        "MAE": svd_mae
    }
])

display(comparison_df)

,Model,Precision@10,Recall@10,NDCG@10,Coverage,RMSE,MAE
0,Popularity,0.001754,0.000111,0.001269,0.004163,NaN,NaN
1,SVD,0.165789,0.042035,0.198545,0.030419,1.014546,0.782679


In [ ]:
movie_genres = movies_df.set_index(
    "movieId"
)["genres"].to_dict()


def get_genre_set(movie_id):
    genres = movie_genres.get(movie_id, "")

    if pd.isna(genres) or genres == "(no genres listed)":
        return set()

    return set(genres.split("|"))


def calculate_diversity(recommendations):

    user_diversities = []

    for movie_list in recommendations.values():

        genre_sets = [
            get_genre_set(movie_id)
            for movie_id in movie_list[:K]
        ]

        distances = []

        for i in range(len(genre_sets)):
            for j in range(i + 1, len(genre_sets)):

                union = genre_sets[i] | genre_sets[j]
                intersection = genre_sets[i] & genre_sets[j]

                if union:
                    distance = 1 - len(intersection) / len(union)
                else:
                    distance = 0.0

                distances.append(distance)

        if distances:
            user_diversities.append(np.mean(distances))

    return np.mean(user_diversities) if user_diversities else 0.0


popularity_diversity = calculate_diversity(
    popularity_recommendations
)

svd_diversity = calculate_diversity(
    svd_recommendations
)

comparison_df.loc[
    comparison_df["Model"] == "Popularity", "Diversity"
] = popularity_diversity

comparison_df.loc[
    comparison_df["Model"] == "SVD", "Diversity"
] = svd_diversity

print("Final model comparison:")
display(comparison_df)

Final model comparison:


,Model,Precision@10,Recall@10,NDCG@10,Coverage,RMSE,MAE,Diversity
0,Popularity,0.001754,0.000111,0.001269,0.004163,NaN,NaN,0.368843
1,SVD,0.165789,0.042035,0.198545,0.030419,1.014546,0.782679,0.700039


In [ ]:
# saving comparison report
evaluation_path = os.path.join(
    REPORT_DIR,
    "model_evaluation_results.csv"
)

comparison_df.to_csv(evaluation_path, index=False)

print("Evaluation results saved:", evaluation_path)

Evaluation results saved: /content/drive/MyDrive/recommendation-pipeline/reports/model_evaluation_results.csv


In [ ]:
mlflow.set_tracking_uri(
    "sqlite:////content/drive/MyDrive/recommendation-pipeline/mlflow.db"
)

mlflow.set_experiment(
    "MovieLens_Recommendation_System"
)

# Log popularity model
with mlflow.start_run(run_name="Popularity_Evaluation"):

    mlflow.log_param("model", "Popularity")
    mlflow.log_param("minimum_ratings", 5)
    mlflow.log_param("K", K)
    mlflow.log_param("relevance_threshold", RELEVANCE_THRESHOLD)

    for metric, value in popularity_metrics.items():
        mlflow.log_metric(metric.replace("@", "_at_"), float(value))

    mlflow.log_metric("Diversity", float(popularity_diversity))

    mlflow.log_artifact(evaluation_path)


# Log SVD model
with mlflow.start_run(run_name="SVD_Evaluation"):

    mlflow.log_param("model", "SVD")
    mlflow.log_param("K", K)
    mlflow.log_param("relevance_threshold", RELEVANCE_THRESHOLD)

    for metric, value in svd_metrics.items():
        mlflow.log_metric(metric.replace("@", "_at_"), float(value))

    mlflow.log_metric("Diversity", float(svd_diversity))
    mlflow.log_metric("RMSE", float(svd_rmse))
    mlflow.log_metric("MAE", float(svd_mae))

    mlflow.log_artifact(evaluation_path)

print("Both model evaluation experiments logged successfully!")

Both model evaluation experiments logged successfully!


In [ ]:
import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/recommendation-pipeline"

results_path = os.path.join(
    PROJECT_DIR,
    "reports",
    "model_evaluation_results.csv"
)

print("Evaluation report exists:", os.path.exists(results_path))

if os.path.exists(results_path):
    results_df = pd.read_csv(results_path)
    display(results_df)

print("\nModel files:")
for filename in ["popularity_model.pkl", "svd_model.pkl"]:
    path = os.path.join(PROJECT_DIR, "models", filename)
    print(filename, ":", os.path.exists(path))

Evaluation report exists: True


,Model,Precision@10,Recall@10,NDCG@10,Coverage,RMSE,MAE,Diversity
0,Popularity,0.001754,0.000111,0.001269,0.004163,NaN,NaN,0.368843
1,SVD,0.165789,0.042035,0.198545,0.030419,1.014546,0.782679,0.700039



Model files:
popularity_model.pkl : True
svd_model.pkl : True


In [ ]:
import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/recommendation-pipeline"

feature_catalog = pd.DataFrame([
    {
        "feature_name": "user_avg_rating",
        "feature_type": "User",
        "description": "Average rating given by a user",
        "source": "Training ratings",
        "calculation": "Mean rating grouped by userId"
    },
    {
        "feature_name": "user_activity_count",
        "feature_type": "User",
        "description": "Number of ratings given by a user",
        "source": "Training ratings",
        "calculation": "Count of ratings grouped by userId"
    },
    {
        "feature_name": "item_avg_rating",
        "feature_type": "Item",
        "description": "Average rating received by a movie",
        "source": "Training ratings",
        "calculation": "Mean rating grouped by movieId"
    },
    {
        "feature_name": "item_popularity",
        "feature_type": "Item",
        "description": "Number of ratings received by a movie",
        "source": "Training ratings",
        "calculation": "Count of ratings grouped by movieId"
    },
    {
        "feature_name": "unique_user_count",
        "feature_type": "Item",
        "description": "Number of distinct users who rated a movie",
        "source": "Training ratings",
        "calculation": "Nunique userId grouped by movieId"
    },
    {
        "feature_name": "year",
        "feature_type": "Interaction",
        "description": "Year of the rating interaction",
        "source": "Rating timestamp",
        "calculation": "Extract year from timestamp"
    },
    {
        "feature_name": "month",
        "feature_type": "Interaction",
        "description": "Month of the rating interaction",
        "source": "Rating timestamp",
        "calculation": "Extract month from timestamp"
    },
    {
        "feature_name": "day",
        "feature_type": "Interaction",
        "description": "Day of the rating interaction",
        "source": "Rating timestamp",
        "calculation": "Extract day from timestamp"
    }
])

catalog_path = os.path.join(
    PROJECT_DIR,
    "reports",
    "feature_catalog.csv"
)

feature_catalog.to_csv(catalog_path, index=False)

print("Feature catalog saved successfully!")
display(feature_catalog)

Feature catalog saved successfully!


,feature_name,feature_type,description,source,calculation
0,user_avg_rating,User,Average rating given by a user,Training ratings,Mean rating grouped by userId
1,user_activity_count,User,Number of ratings given by a user,Training ratings,Count of ratings grouped by userId
2,item_avg_rating,Item,Average rating received by a movie,Training ratings,Mean rating grouped by movieId
3,item_popularity,Item,Number of ratings received by a movie,Training ratings,Count of ratings grouped by movieId
4,unique_user_count,Item,Number of distinct users who rated a movie,Training ratings,Nunique userId grouped by movieId
5,year,Interaction,Year of the rating interaction,Rating timestamp,Extract year from timestamp
6,month,Interaction,Month of the rating interaction,Rating timestamp,Extract month from timestamp
7,day,Interaction,Day of the rating interaction,Rating timestamp,Extract day from timestamp
